# 02 – Fertilitet per kommun

**Syfte:** Göra om SCB:s summerade fruktsamhet per kommun 2000–2025 till en analysfärdig tabell med en rad per kommun och år. Tabellen har kommunkod, hanterat bortfall och ett avtrendat mått.

| | |
|---|---|
| **Källa** | SCB, Statistikdatabasen: summerad fruktsamhet per kommun och kön |
| **Rådata** | `data/raw/FertilitetPerKommun.csv` (oförändrad, identisk med första versionen i git, commit `93f655c`) |
| **Hjälpfiler** | `data/raw/befolkningstathet.csv` (bara för kommunkoder och folkmängd), `data/processed/fruktbarhet_riket.csv` (från `01`) |
| **Resultat** | `data/processed/fertilitet_kommun.csv`, `data/processed/exkluderade_kommuner.csv` |
| **Beroenden** | Kör `01_Fruktbarhet_riket` först. `03_befolkningstathet` läser listan över uteslutna kommuner härifrån. |

### Beslut som genomförs (från dokumentet *Data preprocessing*)

1. **Uteslut kommuner med minst 75 % bortfall** från all analys på kommunnivå. De redovisas i en egen fil och räknas inte som noll.
2. **Behåll övriga kommuner med `NaN` intakt.** Inga nollor och inga gissade värden.
3. **Interpolera linjärt bara enstaka hål**, alltså ett saknat år mellan två kända år, och bara i kommuner med litet bortfall. Varje interpolerat värde flaggas.
4. **Knivsta behålls.** Serien börjar 2003 eftersom kommunen bildades då.
5. **Lägg till flaggor** (`n_ar_med_data`, `har_bortfall`) i stället för att filtrera i tysthet. Då går det att göra en robusthetskontroll med och utan de små kommunerna.
6. **Välj kvinnor** som mått, eftersom det är standardmåttet. Männens serie följer kvinnornas nästan helt, och båda tillsammans skulle ge triviala regler i ARM.
7. **Avtrenda mot riket:** `skillnad_mot_riket = fruktsamhet − rikets fruktsamhet samma år`.

## 1. Inställningar och parametrar

Här samlas besluten som går att justera. Ändras en gräns här slår ändringen igenom i hela notebooken.

- `KON`: vilket kön som används som utfallsmått.
- `GRANS_UTESLUT`: kommuner med minst så stor andel saknade värden utesluts.
- `GRANS_INTERPOLERA`: bara kommuner med *mindre* bortfall än så får sina enstaka hål interpolerade. Kommuner med stort bortfall har så gles data att ett hål inte längre är ett undantag. Där lämnas hålen tomma.

Bortfallsandelen räknas på **båda könen** (26 år × 2 = 52 värden per kommun), på samma sätt som i beslutsdokumentet.

In [1]:
from pathlib import Path

import pandas as pd

# Notebooken körs normalt från mappen Assignment.
# Körs den från projektroten letar vi upp Assignment ändå.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Hittar inte data/raw. Öppna notebooken från mappen Assignment."

KON = "kvinnor"
GRANS_UTESLUT = 0.75
GRANS_INTERPOLERA = 0.25

## 2. Läs in rådata

Filen är semikolonseparerad och kodad i UTF-8 med BOM. Saknade värden står som tomma fält, och dem läser pandas automatiskt som `NaN`. Varje rad är en kommun, ett kön och ett år.

In [2]:
fert_ra = pd.read_csv(RAW / "FertilitetPerKommun.csv", sep=";", encoding="utf-8-sig")

display(fert_ra.head())
print("Form:", fert_ra.shape)
print(fert_ra.dtypes)

,region,kön,tabellinnehåll,år,Värde
0,Upplands Väsby,män,Antal,2000,1.43
1,Upplands Väsby,män,Antal,2001,1.52
2,Upplands Väsby,män,Antal,2002,1.51
3,Upplands Väsby,män,Antal,2003,1.64
4,Upplands Väsby,män,Antal,2004,1.56


Form: (15080, 5)
region                str
kön                   str
tabellinnehåll        str
år                  int64
Värde             float64
dtype: object


## 3. Kontrollera strukturen

Vi förväntar oss **290 kommuner × 2 kön × 26 år = 15 080 rader** och inga dubbletter.

Kolumnen `tabellinnehåll` har bara värdet `Antal`. Det är **missvisande**: värdena är summerad fruktsamhet (barn per person, t.ex. 1,69), inte ett antal födda. Kolumnen innehåller ingen information och tas bort i steg 6.

In [3]:
assert list(fert_ra.columns) == ["region", "kön", "tabellinnehåll", "år", "Värde"]
assert fert_ra["region"].nunique() == 290
assert set(fert_ra["kön"]) == {"män", "kvinnor"}
assert fert_ra["år"].min() == 2000 and fert_ra["år"].max() == 2025
assert len(fert_ra) == 290 * 2 * 26
assert not fert_ra.duplicated(["region", "kön", "år"]).any()
assert fert_ra["tabellinnehåll"].unique().tolist() == ["Antal"]
assert pd.api.types.is_float_dtype(fert_ra["Värde"])

print("Saknade värden totalt:", fert_ra["Värde"].isna().sum(), "av", len(fert_ra),
      f"({fert_ra['Värde'].isna().mean():.1%})")
print("Värdeintervall:", fert_ra["Värde"].min(), "–", fert_ra["Värde"].max())

Saknade värden totalt: 414 av 15080 (2.7%)
Värdeintervall: 0.8 – 3.15


## 4. Koppla på kommunkod

Fertilitetsfilen har bara kommunnamn. Namn är en bräcklig nyckel, så vi hämtar den fyrsiffriga **kommunkoden** från täthetsfilen. Den kommer också från SCB och har både kod och namn. Därefter kan alla tabeller kopplas ihop på kod i stället för namn.

Namnen jämförs efter `strip()` och `casefold()` för att tåla skillnader i mellanslag och versaler. Vi kontrollerar att matchningen är **ett-till-ett** och att varje kommun får en kod. Kommunkoden läses som text, så att inledande nollor behålls (`0114`, inte `114`).

Täthetsfilens folkmängd behålls tillfälligt. Den används i nästa steg för att visa att bortfallet hänger ihop med kommunstorlek.

In [4]:
tathet_ra = pd.read_csv(RAW / "befolkningstathet.csv", encoding="utf-8-sig", dtype={"Kommunkod": str})

nyckel = tathet_ra[["Kommunkod", "Kommun"]].drop_duplicates()
nyckel["namnnyckel"] = nyckel["Kommun"].str.strip().str.casefold()
assert nyckel["Kommunkod"].str.fullmatch(r"\d{4}").all()
assert not nyckel.duplicated("namnnyckel").any()
assert not nyckel.duplicated("Kommunkod").any()

fert = fert_ra.assign(namnnyckel=fert_ra["region"].str.strip().str.casefold())
fert = fert.merge(nyckel, on="namnnyckel", how="left", validate="many_to_one")
assert fert["Kommunkod"].notna().all(), "Något kommunnamn saknar kod i täthetsfilen."

folkmangd_2025 = (
    tathet_ra.loc[tathet_ra["År"] == 2025, ["Kommunkod", "Folkmängd"]]
    .set_index("Kommunkod")["Folkmängd"]
)
print("Alla", fert["Kommunkod"].nunique(), "kommuner fick en kommunkod.")

Alla 290 kommuner fick en kommunkod.


## 5. Bortfall per kommun

Bortfallet är litet totalt (2,7 %), men det är **koncentrerat** till ett fåtal kommuner, och det är **inte slumpmässigt**. Kommuner med bortfall är betydligt mindre än övriga. Det tyder på att SCB inte redovisar talet när underlaget är för litet. Tomrummet säger alltså i sig något om kommunen. Därför fyller vi inte i värden med genomsnitt eller nollor.

Tabellen visar alla kommuner som har något bortfall, sorterade efter andel.

In [5]:
bortfall = (
    fert.groupby(["Kommunkod", "Kommun"])["Värde"]
    .agg(n_varden="size", n_saknas=lambda s: s.isna().sum())
    .reset_index()
)
bortfall["bortfall_andel"] = bortfall["n_saknas"] / bortfall["n_varden"]
bortfall["folkmangd_2025"] = bortfall["Kommunkod"].map(folkmangd_2025).round().astype(int)

med_bortfall = bortfall.loc[bortfall["n_saknas"] > 0].sort_values("bortfall_andel", ascending=False)
display(med_bortfall.assign(bortfall_procent=(med_bortfall["bortfall_andel"] * 100).round(0))
        .drop(columns="bortfall_andel"))

utan = bortfall.loc[bortfall["n_saknas"] == 0, "folkmangd_2025"].median()
med = med_bortfall["folkmangd_2025"].median()
print(f"Kommuner med bortfall: {len(med_bortfall)} av {len(bortfall)}")
print(f"Medianfolkmängd 2025 – med bortfall: {med:,.0f}, utan bortfall: {utan:,.0f}")

,Kommunkod,Kommun,n_varden,n_saknas,folkmangd_2025,bortfall_procent
262,2403,Bjurholm,52,52,2327,100.0
269,2425,Dorotea,52,50,2241,96.0
268,2422,Sorsele,52,46,2362,88.0
279,2513,Överkalix,52,44,3183,85.0
277,2506,Arjeplog,52,40,2570,77.0
272,2463,Åsele,52,40,2683,77.0
188,1762,Munkfors,52,27,3637,52.0
266,2418,Malå,52,22,2902,42.0
281,2518,Övertorneå,52,20,3995,38.0
44,0512,Ydre,52,16,3619,31.0


Kommuner med bortfall: 22 av 290
Medianfolkmängd 2025 – med bortfall: 3,903, utan bortfall: 17,522


## 6. Uteslut kommuner med minst 75 % bortfall

Kommuner där minst tre fjärdedelar av värdena saknas har ingen tidsserie att analysera. Dorotea har till exempel bara 2 av 52 värden. Det som finns kvar är brus från ett mycket litet underlag. Kommunerna tas bort från all analys på kommunnivå.

De tas inte bort i tysthet. Listan sparas i `exkluderade_kommuner.csv` med antal värden och motivering, så att urvalet kan redovisas i metoddelen. `03_befolkningstathet` läser samma lista, så att alla filer utesluter samma kommuner.

In [6]:
exkluderade = bortfall.loc[bortfall["bortfall_andel"] >= GRANS_UTESLUT].copy()
exkluderade["n_ar_med_data"] = exkluderade["n_varden"] - exkluderade["n_saknas"]
exkluderade["motivering"] = (
    f"Minst {GRANS_UTESLUT:.0%} av värdena saknas (båda könen 2000–2025). "
    "För lite data för en tidsserie. Utesluten från analys på kommunnivå."
)
exkluderade = exkluderade[["Kommunkod", "Kommun", "n_ar_med_data", "n_varden", "bortfall_andel", "motivering"]]
exkluderade = exkluderade.sort_values("bortfall_andel", ascending=False)

exkluderade.to_csv(PROCESSED / "exkluderade_kommuner.csv", index=False, encoding="utf-8-sig")
display(exkluderade.drop(columns="motivering").round(2))

fert = fert.loc[~fert["Kommunkod"].isin(exkluderade["Kommunkod"])]
print(f"Uteslöt {len(exkluderade)} kommuner. Kvar: {fert['Kommunkod'].nunique()} kommuner.")

,Kommunkod,Kommun,n_ar_med_data,n_varden,bortfall_andel
262,2403,Bjurholm,0,52,1.00
269,2425,Dorotea,2,52,0.96
268,2422,Sorsele,6,52,0.88
279,2513,Överkalix,8,52,0.85
272,2463,Åsele,12,52,0.77
277,2506,Arjeplog,12,52,0.77


Uteslöt 6 kommuner. Kvar: 284 kommuner.


## 7. Välj kön och städa kolumner

Vi behåller **kvinnornas** fruktsamhet, som är standardmåttet. Männens serie följer kvinnornas nästan helt, så båda tillsammans skulle ge meningslösa regler som *"hög fruktsamhet män → hög fruktsamhet kvinnor"*. Männens värden finns kvar i rådatafilen om de behövs senare.

Kolumnerna döps om till korta ASCII-namn (`ar`, `fruktsamhet`). Tomma eller konstanta kolumner tas bort (`kön`, `tabellinnehåll`, `namnnyckel`). Kommunnamnet tas från täthetsfilen, så att stavningen är densamma i alla bearbetade filer.

In [7]:
fert = (
    fert.loc[fert["kön"] == KON, ["Kommunkod", "Kommun", "år", "Värde"]]
    .rename(columns={"år": "ar", "Värde": "fruktsamhet"})
    .sort_values(["Kommunkod", "ar"])
    .reset_index(drop=True)
)

assert len(fert) == fert["Kommunkod"].nunique() * 26
display(fert.head())
print("Form:", fert.shape, "| saknade värden:", fert["fruktsamhet"].isna().sum())

,Kommunkod,Kommun,ar,fruktsamhet
0,0114,Upplands Väsby,2000,1.54
1,0114,Upplands Väsby,2001,1.59
2,0114,Upplands Väsby,2002,1.65
3,0114,Upplands Väsby,2003,1.81
4,0114,Upplands Väsby,2004,1.69


Form: (7384, 4) | saknade värden: 69


## 8. Knivsta – serien börjar 2003

Knivsta bröts ut ur Uppsala kommun och blev en egen kommun **2003**. Värdena för 2000–2002 saknas därför av en strukturell orsak, inte på grund av litet underlag. Vi kontrollerar att det är just de åren som saknas. Värdena lämnas som `NaN`, så serien börjar i praktiken 2003.

Hålen i Knivsta ligger i början av serien. De kan alltså inte interpoleras i nästa steg, eftersom det inte finns något känt värde före dem.

In [8]:
knivsta = fert.loc[fert["Kommun"] == "Knivsta"]
saknade_ar = knivsta.loc[knivsta["fruktsamhet"].isna(), "ar"].tolist()

assert saknade_ar == [2000, 2001, 2002], saknade_ar
forsta_ar = knivsta.loc[knivsta["fruktsamhet"].notna(), "ar"].min()
assert forsta_ar == 2003
print("Knivsta saknar värden för:", saknade_ar, "– serien börjar", forsta_ar)

Knivsta saknar värden för: [2000, 2001, 2002] – serien börjar 2003


## 9. Flaggor för datatäckning

Två kolumner gör bortfallet synligt i analysen, i stället för att det försvinner i tysthet:

- `n_ar_med_data`: antal år av 26 med ett *observerat* värde för kommunen. Räknas **före** interpolationen.
- `har_bortfall`: `True` om kommunen saknar minst ett år.

Med flaggorna kan analysen köras med och utan de små kommunerna, till exempel `df[~df["har_bortfall"]]`, för att se om reglerna håller.

In [9]:
fert["n_ar_med_data"] = fert.groupby("Kommunkod")["fruktsamhet"].transform("count").astype(int)
fert["har_bortfall"] = fert["n_ar_med_data"] < 26

print("Kommuner med bortfall:", fert.loc[fert["har_bortfall"], "Kommunkod"].nunique())
print("Kommuner utan bortfall:", fert.loc[~fert["har_bortfall"], "Kommunkod"].nunique())

Kommuner med bortfall: 14
Kommuner utan bortfall: 270


## 10. Interpolera enstaka hål

Ett saknat år **mellan två kända år** kan fyllas med linjär interpolation. För ett enstaka år är det medelvärdet av året före och året efter. Det är ingen gissning, eftersom värdet ligger mellan två uppmätta punkter.

Två villkor avgör vad som interpoleras:

1. Hålet är **exakt ett år** långt, och både året före och året efter har värden. Längre hål och hål i början eller slutet av serien lämnas som `NaN`.
2. Kommunens totala bortfall är **under `GRANS_INTERPOLERA`**. I kommuner med stort bortfall är ett hål inte längre ett undantag. Där skulle interpolationen hitta på mer än den fyller i.

Varje ifyllt värde får `interpolerad = True`, så att det går att utesluta i en robusthetskontroll.

In [10]:
foregaende = fert.groupby("Kommunkod")["fruktsamhet"].shift(1)
nasta = fert.groupby("Kommunkod")["fruktsamhet"].shift(-1)
bortfall_andel = fert["Kommunkod"].map(bortfall.set_index("Kommunkod")["bortfall_andel"])

ensamt_hal = fert["fruktsamhet"].isna() & foregaende.notna() & nasta.notna()
far_interpoleras = ensamt_hal & (bortfall_andel < GRANS_INTERPOLERA)

fert["interpolerad"] = far_interpoleras
fert.loc[far_interpoleras, "fruktsamhet"] = (foregaende + nasta)[far_interpoleras] / 2

visning = fert.loc[far_interpoleras, ["Kommun", "ar", "fruktsamhet"]].assign(
    ar_fore=foregaende[far_interpoleras], ar_efter=nasta[far_interpoleras])
display(visning.round(3))

ej = fert.loc[ensamt_hal & ~far_interpoleras, "Kommun"].value_counts()
print(f"Interpolerade: {far_interpoleras.sum()} värden i {fert.loc[far_interpoleras, 'Kommun'].nunique()} kommuner.")
print(f"Enstaka hål som lämnas tomma på grund av stort bortfall: {ensamt_hal.sum() - far_interpoleras.sum()}",
      ej.to_dict())
print("Kvar som NaN totalt:", fert["fruktsamhet"].isna().sum())

,Kommun,ar,fruktsamhet,ar_fore,ar_efter
3741,Dals-Ed,2023,1.685,1.55,1.82
3967,Gullspång,2015,2.230,2.15,2.31
4847,Storfors,2011,2.535,2.55,2.52
4851,Storfors,2015,2.155,2.47,1.84
4858,Storfors,2022,1.915,1.98,1.85
5223,Laxå,2023,1.780,1.78,1.78
5316,Ljusnarsberg,2012,2.155,2.19,2.12
5497,Skinnskatteberg,2011,2.155,2.13,2.18
5507,Skinnskatteberg,2021,1.840,1.81,1.87
6602,Ragunda,2024,1.690,1.98,1.40


Interpolerade: 11 värden i 8 kommuner.
Enstaka hål som lämnas tomma på grund av stort bortfall: 15 {'Övertorneå': 5, 'Ydre': 4, 'Munkfors': 3, 'Malå': 3}
Kvar som NaN totalt: 58


## 11. Avtrenda mot riket

Fruktsamheten har en tydlig **nationell trend**: den steg till omkring 2010 och har sjunkit sedan dess. Om de absoluta värdena delas in i klasser över hela perioden blir "hög fruktsamhet" nästan samma sak som "åren kring 2010". ARM skulle då bara hitta kalendern.

Lösningen är att jämföra varje kommun med **riket samma år**:

`skillnad_mot_riket = fruktsamhet − rikets fruktsamhet`

Ett positivt värde betyder fler barn per kvinna än riket samma år. Den nationella trenden försvinner, och den kommunala skillnaden blir kvar.

Referensen är **SCB:s officiella rikstal** (från `01`), inte ett medelvärde av kommunerna. Ett oviktat kommunmedelvärde ger lilla Bjurholm samma vikt som Stockholm och hamnar systematiskt för högt, vilket tabellen nedan visar. Med den basen skulle de flesta kommuner felaktigt hamna under riket.

In [11]:
riket = pd.read_csv(PROCESSED / "fruktbarhet_riket.csv", encoding="utf-8-sig")
assert riket["ar"].tolist() == list(range(2000, 2026)), "Kör 01_Fruktbarhet_riket först."

jamforelse = riket[["ar", "fruktsamhet_riket_kvinnor"]].assign(
    medel_av_kommuner=riket["ar"].map(fert.groupby("ar")["fruktsamhet"].mean()))
jamforelse["skillnad"] = jamforelse["medel_av_kommuner"] - jamforelse["fruktsamhet_riket_kvinnor"]
display(jamforelse.loc[jamforelse["ar"].isin([2000, 2010, 2020, 2025])].round(2))

fert = fert.merge(
    riket[["ar", "fruktsamhet_riket_kvinnor"]].rename(columns={"fruktsamhet_riket_kvinnor": "fruktsamhet_riket"}),
    on="ar", how="left", validate="many_to_one",
)
assert fert["fruktsamhet_riket"].notna().all()
fert["skillnad_mot_riket"] = fert["fruktsamhet"] - fert["fruktsamhet_riket"]

,ar,fruktsamhet_riket_kvinnor,medel_av_kommuner,skillnad
0,2000,1.54,1.69,0.15
10,2010,1.98,2.11,0.13
20,2020,1.66,1.84,0.18
25,2025,1.42,1.57,0.15


### Kontroll: försvann trenden?

Vi jämför kommunernas **median per år** före och efter avtrendningen. Före rör sig medianen tydligt med den nationella trenden. Efter ska den ligga nästan stilla. Spridningen mellan åren (standardavvikelsen av årsmedianerna) ska därför bli mycket mindre.

Att medianen efter avtrendningen ligger runt **+0,12 och inte 0** är väntat. Rikstalet domineras av de folkrika kommunerna, och storstäderna har lägre fruktsamhet än genomsnittskommunen. Den typiska kommunen ligger därför lite över riket. Det viktiga är att nivån är **stabil över tid**, inte att den är noll.

In [12]:
arsmedian = fert.groupby("ar")[["fruktsamhet", "skillnad_mot_riket"]].median()

display(arsmedian.loc[[2000, 2005, 2010, 2015, 2020, 2025]].round(2))
print("Spridning mellan årsmedianer (std):")
print(f"  fruktsamhet (rå):   {arsmedian['fruktsamhet'].std():.3f}")
print(f"  skillnad_mot_riket: {arsmedian['skillnad_mot_riket'].std():.3f}")

,fruktsamhet,skillnad_mot_riket
ar,,
2000,1.66,0.12
2005,1.91,0.14
2010,2.10,0.12
2015,1.97,0.12
2020,1.81,0.15
2025,1.54,0.12


Spridning mellan årsmedianer (std):
  fruktsamhet (rå):   0.159
  skillnad_mot_riket: 0.021


## 12. Slutkontroll

Innan filen sparas kontrollerar vi att:

- varje kommun har exakt 26 rader (2000–2025) och att nyckeln `Kommunkod` + `ar` är unik
- inga uteslutna kommuner finns kvar
- det saknade värdet i `skillnad_mot_riket` finns exakt där fruktsamheten saknas, och ingen annanstans.

In [13]:
kolumner = ["Kommunkod", "Kommun", "ar", "fruktsamhet", "interpolerad", "fruktsamhet_riket",
            "skillnad_mot_riket", "n_ar_med_data", "har_bortfall"]
fert = fert[kolumner]

assert not fert.duplicated(["Kommunkod", "ar"]).any()
assert (fert.groupby("Kommunkod").size() == 26).all()
assert not fert["Kommunkod"].isin(exkluderade["Kommunkod"]).any()
assert fert["skillnad_mot_riket"].isna().equals(fert["fruktsamhet"].isna())

print(f"{fert['Kommunkod'].nunique()} kommuner × 26 år = {len(fert)} rader")
print(f"Saknade värden kvar: {fert['fruktsamhet'].isna().sum()} ({fert['fruktsamhet'].isna().mean():.1%})")
print(f"Interpolerade värden: {fert['interpolerad'].sum()}")
display(fert.head())

284 kommuner × 26 år = 7384 rader
Saknade värden kvar: 58 (0.8%)
Interpolerade värden: 11


,Kommunkod,Kommun,ar,fruktsamhet,interpolerad,fruktsamhet_riket,skillnad_mot_riket,n_ar_med_data,har_bortfall
0,0114,Upplands Väsby,2000,1.54,False,1.54,0.00,26,False
1,0114,Upplands Väsby,2001,1.59,False,1.57,0.02,26,False
2,0114,Upplands Väsby,2002,1.65,False,1.65,0.00,26,False
3,0114,Upplands Väsby,2003,1.81,False,1.71,0.10,26,False
4,0114,Upplands Väsby,2004,1.69,False,1.75,-0.06,26,False


## 13. Spara

Två filer sparas:

- `fertilitet_kommun.csv` – analysunderlaget, en rad per kommun och år.
- `exkluderade_kommuner.csv` – sparades redan i steg 6 och används av `03_befolkningstathet`.

Kommunkoden sparas som text med inledande nolla. Läs därför in filen med `dtype={"Kommunkod": str}`.

In [14]:
utfil = PROCESSED / "fertilitet_kommun.csv"
fert.to_csv(utfil, index=False, encoding="utf-8-sig")

print(f"Sparade {len(fert)} rader till {utfil.relative_to(BAS)}")
print(f"Sparade {len(exkluderade)} uteslutna kommuner till data/processed/exkluderade_kommuner.csv")

Sparade 7384 rader till data/processed/fertilitet_kommun.csv
Sparade 6 uteslutna kommuner till data/processed/exkluderade_kommuner.csv
